# Workshop 1 — Tensors

## Part 1 — Conda
Let's first create a conda environment which we'll use throughout the duration of these 12 workshop sessions. Either use the `environment.yml` file provided, or, you can create your own using the following.

```bash
# Create a new environment named 'kunming-ai' with Python 3.11
conda create -n kunming-ai python=3.11 -y

# Activate it
conda activate kunming-ai

# Install packages 
pip install torch torchvision matplotlib numpy

# Leave the environment
conda deactivate

```

### Quick sanity check

Run this cell. If you see a PyTorch version printed, then everything is set.

In [1]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.5.1
CUDA available: True


## Part 2 — Tensors
A tensor is just a multi-dimensional array of numbers:

- A **scalar** is a 0-D tensor: `7`
- A **vector** is a 1-D tensor: `[1, 2, 3]`
- A **matrix** is a 2-D tensor: `[[1, 2], [3, 4]]`
- An **image** is often a 3-D tensor: `(channels, height, width)`
- A **batch of images** is a 4-D tensor: `(batch, channels, height, width)`

In most instances, NumPy is a package which is used for dealing with tensor data. However, for machine learning related tasks, we instead use PyTorch.

Everything in PyTorch is a **tensor**. Tensors in PyTorch have two major advantages over NumPy:
1. They can live on a **GPU**, which makes operations massively faster.
2. They can track **gradients** automatically.

### 2.1 Creating tensors

The most common ways to create a tensor:

In [2]:
# 1. From a Python list
example_list = [1, 2, 3]
a = torch.tensor(example_list)

# 2. All zeros
b = torch.zeros(3, 4)

# 3. All ones
c = torch.ones(3, 4)

# 4. Random (uniform [0,1))
d = torch.rand(3, 4)

# 5. Range from 0 to 9
e = torch.arange(0, 10, 1)

print("a (from Python list):\n", a, "\n" + "-"*40)
print("b (all zeros, shape 3x4):\n", b, "\n" + "-"*40)
print("c (all ones, shape 3x4):\n", c, "\n" + "-"*40)
print("d (random, uniform [0,1), shape 3x4):\n", d, "\n" + "-"*40)
print("e (range from 0 to 9):\n", e, "\n" + "-"*40)


a (from Python list):
 tensor([1, 2, 3]) 
----------------------------------------
b (all zeros, shape 3x4):
 tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]]) 
----------------------------------------
c (all ones, shape 3x4):
 tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]]) 
----------------------------------------
d (random, uniform [0,1), shape 3x4):
 tensor([[0.3052, 0.4747, 0.2817, 0.1053],
        [0.0182, 0.7432, 0.9653, 0.9203],
        [0.5978, 0.2863, 0.0871, 0.2274]]) 
----------------------------------------
e (range from 0 to 9):
 tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]) 
----------------------------------------


If you have data which is in the form of a NumPy array, you can very easily convert this into a PyTorch tensor.

In [3]:
import numpy as np

a = np.array([1, 2, 3])
b = torch.from_numpy(a)

print(f"Original NumPy array (a): {a}")
print(f"Converted to PyTorch tensor (b): {b}")

# You can also convert a PyTorch tensor back into a NumPy array.
c = b.numpy()

print(f"Tensor converted back to NumPy array (c): {c}")

Original NumPy array (a): [1 2 3]
Converted to PyTorch tensor (b): tensor([1, 2, 3])
Tensor converted back to NumPy array (c): [1 2 3]


### 2.2 Shape, dtype, and device

Every tensor carries three pieces of metadata you should always be able to read off:

- `.shape` — the dimensions
- `.dtype` — the data type (e.g. `torch.float32`, `torch.int64`)
- `.device` — where it lives (`cpu` or `cuda:0`)

It is very useful to print these especially during development stages as most PyTorch related bugs will come from shape mismatch or a dtype mismatch.

In [4]:
x = torch.rand(2, 3, 4)

print("shape:", x.shape)
print("dtype:", x.dtype)
print("device:", x.device)

shape: torch.Size([2, 3, 4])
dtype: torch.float32
device: cpu


### 2.3 Indexing and slicing

Indexing is an operation that allows you to pull out single elements, rows, columns, or slabs.

In [6]:
m = torch.arange(20).reshape(4, 5)

print("Tensor m (shape: {}):\n{}\n".format(tuple(m.shape), m))

# 1. Element at row 2, column 3
r2c3 = m[2, 3]

print(f"Element at row 2, column 3: m[2, 3] = {r2c3.item()}\n")

# 2. Second row (index 1)
r1 = m[1]

print(f"Second row (m[1]):\n{r1}\n")

# 3. Third column (index 2)
c2 = m[:, 2]

print(f"Third column (m[:, 2]):\n{c2}\n")

# 4. Top-left 2x2 submatrix (m[:2, :2]):
t2x2 = m[:2, :2]

print(f"Top-left 2x2 submatrix (m[:2, :2]):\n{t2x2}\n")


Tensor m (shape: (4, 5)):
tensor([[ 0,  1,  2,  3,  4],
        [ 5,  6,  7,  8,  9],
        [10, 11, 12, 13, 14],
        [15, 16, 17, 18, 19]])

Element at row 2, column 3: m[2, 3] = 13

Second row (m[1]):
tensor([5, 6, 7, 8, 9])

Third column (m[:, 2]):
tensor([ 2,  7, 12, 17])

Top-left 2x2 submatrix (m[:2, :2]):
tensor([[0, 1],
        [5, 6]])



### 2.4 Reshaping

Reshaping changes how the tensor is *viewed*, not the underlying numbers. The total number of elements must stay the same.

- `.reshape(new_shape)` — most common
- `.view(new_shape)` — like reshape but requires contiguous memory
- `.unsqueeze(dim)` — add a size-1 dimension at position `dim`
- `.squeeze()` — remove all size-1 dimensions

The `-1` trick: use `-1` in one position to mean "figure it out from the others".

In [11]:
v = torch.arange(12)

print(f"Original v: {v.tolist()} (shape: {tuple(v.shape)})\n")

# 1. 3x4
v_3x4 = v.reshape(3, 4)

print("Reshaped to 3x4:\n", v_3x4, f"\nshape: {tuple(v_3x4.shape)}\n")

# 2. 2x2x3
v_2x2x3 = v.reshape(2, 2, 3)

print("Reshaped to 2x2x3:\n", v_2x2x3, f"\nshape: {tuple(v_2x2x3.shape)}\n")

# 3. -1 inferred
v__inf_6 = v.reshape(-1, 6)

print("Reshaped with -1 (infer rows), (-1, 6):\n", v__inf_6, f"\nshape: {tuple(v__inf_6.shape)}\n")

# 4. Add a leading dim
w = torch.arange(5)
w_unsq_0 = w.unsqueeze(0)

print(f"Original w: {w.tolist()} (shape: {tuple(w.shape)})")
print("w.unsqueeze(0):", w_unsq_0, f"shape: {tuple(w_unsq_0.shape)}\n")

Original v: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] (shape: (12,))

Reshaped to 3x4:
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]]) 
shape: (3, 4)

Reshaped to 2x2x3:
 tensor([[[ 0,  1,  2],
         [ 3,  4,  5]],

        [[ 6,  7,  8],
         [ 9, 10, 11]]]) 
shape: (2, 2, 3)

Reshaped with -1 (infer rows), (-1, 6):
 tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11]]) 
shape: (2, 6)

Original w: [0, 1, 2, 3, 4] (shape: (5,))
w.unsqueeze(0): tensor([[0, 1, 2, 3, 4]]) shape: (1, 5)



An additional operation is `.cat()` and `.stack()`.

- `.cat()` — concatenates a sequence of tensors along a given dimension
- `.stack()` — concatenate a sequence of tensors along a new dimension

In [27]:
a = torch.tensor([[1, 2], [3, 4]])
b = torch.tensor([[5, 6], [7, 8]])


cat_ab_0 = torch.cat([a, b], dim=0)
cat_ab_1 = torch.cat([a, b], dim=1)

print("torch.cat([a, b], dim=0):\n", cat_ab_0, "\n")
print("torch.cat([a, b], dim=1):\n", cat_ab_1, "\n")

stack_ab_0 = torch.stack([a, b], dim=0)
stack_ab_1 = torch.stack([a, b], dim=1)

print("torch.stack([a, b], dim=0):\n", stack_ab_0, "\n")
print("torch.stack([a, b], dim=1):\n", stack_ab_1, "\n")

torch.cat([a, b], dim=0):
 tensor([[1, 2],
        [3, 4],
        [5, 6],
        [7, 8]]) 

torch.cat([a, b], dim=1):
 tensor([[1, 2, 5, 6],
        [3, 4, 7, 8]]) 

torch.stack([a, b], dim=0):
 tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]]) 

torch.stack([a, b], dim=1):
 tensor([[[1, 2],
         [5, 6]],

        [[3, 4],
         [7, 8]]]) 



### 2.5 Math operations and broadcasting

Element-wise math just works. The interesting part is broadcasting. When shapes don't match exactly, PyTorch can stretch smaller tensors to fit.

The rule (read right-to-left): two dimensions are compatible if they are **equal**, or one of them is **1**.

Examples:
- `(3, 4) + (4,)` → the `(4,)` is broadcast to every row → result `(3, 4)` ✓
- `(3, 4) + (3, 1)` → the column is broadcast across all 4 columns → result `(3, 4)` ✓
- `(3, 4) + (3,)` → ✗ error: trailing dims don't match

In [26]:
# Make a 3x4 tensor
A = torch.arange(12).reshape(3, 4).float()

print("A (original):\n", A, f"\nshape: {tuple(A.shape)}\n")

# 1. Scalar broadcast: A + 10
scalar_broadcast = A + 10
print("A + 10 (add 10 to every element):\n", scalar_broadcast, "\n")

# 2. Row vector broadcast: shape (4,) broadcasts across rows
row_vec = torch.tensor([1, 2, 3, 4])
row_vec_broadcast = A + row_vec

print(f"A + {row_vec.tolist()} (row vector broadcast):\n", row_vec_broadcast, "\n")

# 3. Column vector broadcast: shape (3,1) broadcasts across columns
col_vec = torch.tensor([[10], [20], [30]])
col_vec_broadcast = A + col_vec

print(f"A + {col_vec.tolist()} (column vector broadcast):\n", col_vec_broadcast, "\n")

# 4. Element-wise multiply
element_wise_multiply = A * 2

print("A * 2 (every element multiplied by 2):\n", element_wise_multiply, "\n")

A (original):
 tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]]) 
shape: (3, 4)

A + 10 (add 10 to every element):
 tensor([[10., 11., 12., 13.],
        [14., 15., 16., 17.],
        [18., 19., 20., 21.]]) 

A + [1, 2, 3, 4] (row vector broadcast):
 tensor([[ 1.,  3.,  5.,  7.],
        [ 5.,  7.,  9., 11.],
        [ 9., 11., 13., 15.]]) 

A + [[10], [20], [30]] (column vector broadcast):
 tensor([[10., 11., 12., 13.],
        [24., 25., 26., 27.],
        [38., 39., 40., 41.]]) 

A * 2 (every element multiplied by 2):
 tensor([[ 0.,  2.,  4.,  6.],
        [ 8., 10., 12., 14.],
        [16., 18., 20., 22.]]) 



### 2.6 Matrix multiplication

Two ways:
- `@` (the operator)
- `torch.matmul(A, B)`

For 2-D tensors this is plain matrix multiplication. Shape rule: `(m, k) @ (k, n) → (m, n)`.

In [16]:
A = torch.tensor([[1., 2.], [3., 4.], [5., 6.]])   # 3x2
B = torch.tensor([[1., 0., -1.], [2., 1.,  0.]])   # 2x3

C = A @ B
print("A (3x2):\n", A, f"\nshape: {tuple(A.shape)}\n")
print("B (2x3):\n", B, f"\nshape: {tuple(B.shape)}\n")
print("C = A @ B (3x3):\n", C, f"\nshape: {tuple(C.shape)}")

A (3x2):
 tensor([[1., 2.],
        [3., 4.],
        [5., 6.]]) 
shape: (3, 2)

B (2x3):
 tensor([[ 1.,  0., -1.],
        [ 2.,  1.,  0.]]) 
shape: (2, 3)

C = A @ B (3x3):
 tensor([[ 5.,  2., -1.],
        [11.,  4., -3.],
        [17.,  6., -5.]]) 
shape: (3, 3)


### 2.7 Reductions

Operations that collapse one or more dimensions: `.sum()`, `.mean()`, `.max()`, `.min()`, `.std()`.

By default they reduce over **all** elements. Pass `dim=` to reduce along a specific axis.

In [17]:
X = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])

# 1. Sum of everything
sum_all = X.sum()

print(f"Sum of all elements in X: {sum_all.item()}")

# 2. Mean per row: collapse dim=1 (columns)
row_means = X.mean(dim=1)

print("Mean of each row in X:")
for i, mean in enumerate(row_means.tolist()):
    print(f"  Row {i}: {mean:.2f}")

# 3. Max per column: collapse dim=0 (rows). .max returns (values, indices)
vals, idx = X.max(dim=0)

print("Max value of each column in X:")
for j, (val, ind) in enumerate(zip(vals.tolist(), idx.tolist())):
    print(f"  Col {j}: {val:.2f} (from row {ind})")


Sum of all elements in X: 21.0
Mean of each row in X:
  Row 0: 2.00
  Row 1: 5.00
Max value of each column in X:
  Col 0: 4.00 (from row 1)
  Col 1: 5.00 (from row 1)
  Col 2: 6.00 (from row 1)


### 2.8 The GPU superpower

Tensors can live on a GPU. Operations on GPU tensors are massively parallel — often 10x-100x faster for the kinds of matrix multiplies neural networks do.

The basic pattern:
1. Pick a device (use a GPU if you have one, else fall back to CPU).
2. Move your tensors there with `.to(device)`.
3. Everything you do with them happens on the GPU.

**Rule:** all tensors in an operation must be on the same device, or you'll get an error.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

x = torch.rand(2, 3)
x_device = x.to(device)

print("x.device:", x_device)

x.device: tensor([[0.0237, 0.2510, 0.0491],
        [0.1424, 0.6370, 0.9332]], device='cuda:0')


### 2.9 Feel the speedup

Let's actually see how much faster matrix multiplication is on a GPU.

In [25]:
import time

N = 4000

# CPU
A_cpu = torch.rand(N, N)
B_cpu = torch.rand(N, N)

t0 = time.time()

C_cpu = A_cpu @ B_cpu

t_cpu = time.time() - t0
print(f"CPU: {t_cpu:.3f} s")

# GPU (if available)
if torch.cuda.is_available():
    A_gpu = torch.rand(N, N, device="cuda")
    B_gpu = torch.rand(N, N, device="cuda")
    # warmup
    _ = A_gpu @ B_gpu
    torch.cuda.synchronize()
    t0 = time.time()
    C_gpu = A_gpu @ B_gpu
    torch.cuda.synchronize()        # important: wait for the GPU to finish
    t_gpu = time.time() - t0
    print(f"GPU: {t_gpu:.3f} s")
    print(f"Speedup: {t_cpu / t_gpu:.1f}x")
else:
    print("No GPU available — try a Colab GPU runtime.")


CPU: 0.094 s
GPU: 0.002 s
Speedup: 39.4x


## Recap

You now know:

- **Conda** isolates project dependencies so they don't fight.
- **Tensors** are the unit of computation in PyTorch.
- You can create, index, reshape, do math, and broadcast.
- Matrix multiplication is `@`.
- Reductions collapse axes with `dim=`.
- GPU tensors compute the same operations, but much faster.

### Next: Practical Workshop 1 — *Autograd from scratch*
We will build a neural network using only tensors and automatic differentiation.